# MS3 — Step 9: Tissue-Compartment HLA Expression (GTEx + Human Protein Atlas)

Extracts HLA gene/protein expression across lung, lymph node, spleen, and upper airway
using GTEx (bulk RNA-seq, median TPM) and the Human Protein Atlas (immunohistochemistry,
protein level).

**FRAMING, stated explicitly because it matters**: this is **tissue-compartment
expression variation**, not spatial transcriptomics. GTEx reports one median value per
gene per whole tissue (no within-tissue spatial coordinates); HPA reports
immunohistochemistry staining level per tissue (cell-type-resolved in some tissues, but
still not spatially mapped across a section the way Visium data is). The word "spatial"
should not appear anywhere in results/discussion text describing this step's output.

**GTEx release**: this notebook targets GTEx's **current** release via the API V2
(`https://gtexportal.org/api/v2`), not v10. GTEx API V1 has been discontinued entirely,
and the v10 data release was never available on API V1 to begin with -- the current
adult GTEx release as of this pipeline's construction is **V11** (Jan 2026). Per an
explicit decision earlier in this project, this step targets current data rather than
a specific historical version number, and the methodology text should say so plainly
(e.g. "GTEx (current release, V11 as of this analysis)") rather than hardcoding "v10"
as originally planned.

**HPA tissue names are discovered at runtime, not assumed.** An earlier draft of this
step guessed HPA tissue-name mappings (e.g. "nasopharynx", "minor salivary gland" as
upper-airway proxies) without confirming these are real values in HPA's own vocabulary.
This notebook instead downloads the real `normal_tissue.tsv` file and inspects its
actual `Tissue` column values directly, selecting the best-matching real tissue names
for lung, lymph node, spleen, and upper airway rather than trusting a guess.

**Workflow:**
1. Setup + resolve HLA gene symbols to versioned GENCODE IDs (Part A)
2. Query GTEx API V2 for median expression across target tissues (Part B)
3. Download HPA data, discover real tissue vocabulary, extract HLA protein levels (Part C)
4. Merge, visualize, and export (Part D)

## Part A — Setup and GENCODE ID Resolution

In [ ]:
!pip install -q requests pandas matplotlib seaborn 2>/dev/null || \
  pip install -q --break-system-packages requests pandas matplotlib seaborn

from pathlib import Path
from typing import Optional
import io
import zipfile

import requests
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns

BASE_DIR = Path("/content/ms3_step9_workspace")
TABLES_DIR = BASE_DIR / "tables"
FIGURES_DIR = BASE_DIR / "figures"
for d in (BASE_DIR, TABLES_DIR, FIGURES_DIR):
    d.mkdir(parents=True, exist_ok=True)

GTEX_BASE = "https://gtexportal.org/api/v2"
HPA_URL = "https://www.proteinatlas.org/download/normal_tissue.tsv.zip"

HLA_GENES = ["HLA-A", "HLA-B", "HLA-C", "HLA-DRA", "HLA-DRB1", "HLA-DPA1", "HLA-DPB1", "HLA-E"]

# Target tissue compartments per the manuscript's design
TARGET_COMPARTMENTS = ["lung", "lymph_node", "spleen", "upper_airway"]

FRAMING_NOTE = (
    "HLA expression data sourced from GTEx (bulk RNA-seq, median TPM per tissue, "
    "current release as of this analysis) and Human Protein Atlas (independent "
    "tissue RNA-seq, assayType='tissue', nTPM/pTPM/TPM per tissue). HPA protein "
    "IHC was found unavailable in HPA's current export surface (absent from both "
    "bulk TSV and per-gene XML), so Part C is a same-modality, cross-pipeline "
    "RNA-seq replication of Part B, not a protein-vs-RNA cross-modality check. "
    "Analysis represents tissue-compartment expression variation, not spatial "
    "transcriptomics."
)

print(f"Working directory: {BASE_DIR}")
print(FRAMING_NOTE)

In [2]:
def resolve_gencode_id(gene_symbol: str) -> Optional[dict]:
    """
    Resolve a gene symbol to its versioned GENCODE ID via GTEx's own reference
    endpoint, rather than hardcoding version suffixes (which change across
    GENCODE releases and would silently fail if guessed wrong).
    """
    url = f"{GTEX_BASE}/reference/gene"
    try:
        r = requests.get(url, params={"geneId": gene_symbol}, timeout=20)
        r.raise_for_status()
        data = r.json().get("data", [])
        if not data:
            return None
        # Prefer an exact symbol match if multiple entries come back
        exact = [d for d in data if d.get("geneSymbol", "").upper() == gene_symbol.upper()]
        return (exact or data)[0]
    except requests.RequestException as e:
        print(f"  [WARN] Could not resolve {gene_symbol}: {e}")
        return None

gene_resolution = {}
for gene in HLA_GENES:
    result = resolve_gencode_id(gene)
    if result:
        gencode_id = result.get("gencodeId")
        gene_resolution[gene] = gencode_id
        print(f"  {gene} -> {gencode_id}")
    else:
        gene_resolution[gene] = None
        print(f"  [MISSING] {gene}: could not resolve a GENCODE ID")

unresolved = [g for g, v in gene_resolution.items() if v is None]
if unresolved:
    print(f"\n[WARNING] {len(unresolved)} gene(s) could not be resolved: {unresolved}. "
          f"These will be skipped in Part B.")

  HLA-A -> ENSG00000206503.12
  HLA-B -> ENSG00000234745.10
  HLA-C -> ENSG00000204525.16
  HLA-DRA -> ENSG00000204287.13
  HLA-DRB1 -> ENSG00000196126.11
  HLA-DPA1 -> ENSG00000231389.7
  HLA-DPB1 -> ENSG00000223865.10
  HLA-E -> ENSG00000204592.8


In [4]:
# Diagnostic: show exactly what Part A resolved, and inspect one raw API
# response directly rather than only checking parsed row counts
print("Gene resolution from Part A:")
for gene, gencode_id in gene_resolution.items():
    print(f"  {gene}: {gencode_id}")

print("\n--- Raw API response for HLA-A ---")
test_gencode_id = gene_resolution.get("HLA-A")
if test_gencode_id is None:
    print("[BLOCKED] HLA-A was never resolved in Part A -- check that cell's output above.")
else:
    url = f"{GTEX_BASE}/expression/clusteredMedianGeneExpression"
    r = requests.get(url, params={"gencodeId": test_gencode_id}, timeout=30)
    print(f"Status code: {r.status_code}")
    print(f"URL actually requested: {r.url}")
    print(f"Raw response (first 1000 chars):")
    print(r.text[:1000])

Gene resolution from Part A:
  HLA-A: ENSG00000206503.12
  HLA-B: ENSG00000234745.10
  HLA-C: ENSG00000204525.16
  HLA-DRA: ENSG00000204287.13
  HLA-DRB1: ENSG00000196126.11
  HLA-DPA1: ENSG00000231389.7
  HLA-DPB1: ENSG00000223865.10
  HLA-E: ENSG00000204592.8

--- Raw API response for HLA-A ---
Status code: 200
URL actually requested: https://gtexportal.org/api/v2/expression/clusteredMedianGeneExpression?gencodeId=ENSG00000206503.12
Raw response (first 1000 chars):
{"clusters":null,"medianGeneExpression":[]}


In [6]:
# ============================================================
# Part A — last cell (unchanged; included for copy-paste continuity)
# ============================================================
def resolve_gencode_id(gene_symbol: str) -> Optional[dict]:
    """
    Resolve a gene symbol to its versioned GENCODE ID via GTEx's own reference
    endpoint, rather than hardcoding version suffixes (which change across
    GENCODE releases and would silently fail if guessed wrong).
    """
    url = f"{GTEX_BASE}/reference/gene"
    try:
        r = requests.get(url, params={"geneId": gene_symbol}, timeout=20)
        r.raise_for_status()
        data = r.json().get("data", [])
        if not data:
            return None
        # Prefer an exact symbol match if multiple entries come back
        exact = [d for d in data if d.get("geneSymbol", "").upper() == gene_symbol.upper()]
        return (exact or data)[0]
    except requests.RequestException as e:
        print(f"  [WARN] Could not resolve {gene_symbol}: {e}")
        return None

gene_resolution = {}
for gene in HLA_GENES:
    result = resolve_gencode_id(gene)
    if result:
        gencode_id = result.get("gencodeId")
        gene_resolution[gene] = gencode_id
        print(f"  {gene} -> {gencode_id}")
    else:
        gene_resolution[gene] = None
        print(f"  [MISSING] {gene}: could not resolve a GENCODE ID")

unresolved = [g for g, v in gene_resolution.items() if v is None]
if unresolved:
    print(f"\n[WARNING] {len(unresolved)} gene(s) could not be resolved: {unresolved}. "
          f"These will be skipped in Part B.")


# ============================================================
# Part B — first cell (FIXED: correct response key + pinned datasetId)
# ============================================================
_required = ["gene_resolution", "GTEX_BASE"]
_missing = [name for name in _required if name not in dir()]
if _missing:
    raise RuntimeError(f"Missing: {_missing}. Run Part A first.")

def query_gtex_expression(gencode_id: str) -> list:
    """Query median expression for one gene across all GTEx tissues."""
    url = f"{GTEX_BASE}/expression/clusteredMedianGeneExpression"
    try:
        r = requests.get(
            url,
            params={"gencodeId": gencode_id, "datasetId": "gtex_v8"},  # pin release
            timeout=30,
        )
        r.raise_for_status()
        return r.json().get("medianGeneExpression", [])  # correct key, not "data"
    except requests.RequestException as e:
        print(f"  [WARN] Query failed for {gencode_id}: {e}")
        return []

gtex_rows = []
for gene, gencode_id in gene_resolution.items():
    if gencode_id is None:
        continue
    entries = query_gtex_expression(gencode_id)
    print(f"  {gene}: {len(entries)} tissue entries returned")
    for e in entries:
        gtex_rows.append({
            "gene": gene,
            "gencode_id": gencode_id,
            "tissue_site_detail_id": e.get("tissueSiteDetailId"),
            "median_tpm": e.get("median"),
            "dataset_id": e.get("datasetId"),
        })

gtex_df = pd.DataFrame(gtex_rows)
print(f"\nTotal GTEx entries: {len(gtex_df)}")

if not gtex_df.empty:
    print(f"\nActual tissue site IDs available (sample): "
          f"{sorted(gtex_df['tissue_site_detail_id'].unique())[:15]}")
    print(f"\nDataset ID(s) returned (confirms which GTEx release this data is from): "
          f"{gtex_df['dataset_id'].unique().tolist()}")

gtex_df.head(10)

  HLA-A -> ENSG00000206503.12
  HLA-B -> ENSG00000234745.10
  HLA-C -> ENSG00000204525.16
  HLA-DRA -> ENSG00000204287.13
  HLA-DRB1 -> ENSG00000196126.11
  HLA-DPA1 -> ENSG00000231389.7
  HLA-DPB1 -> ENSG00000223865.10
  HLA-E -> ENSG00000204592.8
  HLA-A: 54 tissue entries returned
  HLA-B: 54 tissue entries returned
  HLA-C: 54 tissue entries returned
  HLA-DRA: 54 tissue entries returned
  HLA-DRB1: 54 tissue entries returned
  HLA-DPA1: 54 tissue entries returned
  HLA-DPB1: 54 tissue entries returned
  HLA-E: 54 tissue entries returned

Total GTEx entries: 432

Actual tissue site IDs available (sample): ['Adipose_Subcutaneous', 'Adipose_Visceral_Omentum', 'Adrenal_Gland', 'Artery_Aorta', 'Artery_Coronary', 'Artery_Tibial', 'Bladder', 'Brain_Amygdala', 'Brain_Anterior_cingulate_cortex_BA24', 'Brain_Caudate_basal_ganglia', 'Brain_Cerebellar_Hemisphere', 'Brain_Cerebellum', 'Brain_Cortex', 'Brain_Frontal_Cortex_BA9', 'Brain_Hippocampus']

Dataset ID(s) returned (confirms which GTEx 

,gene,gencode_id,tissue_site_detail_id,median_tpm,dataset_id
0,HLA-A,ENSG00000206503.12,Adipose_Subcutaneous,1079.350,gtex_v8
1,HLA-A,ENSG00000206503.12,Adipose_Visceral_Omentum,900.181,gtex_v8
2,HLA-A,ENSG00000206503.12,Adrenal_Gland,919.470,gtex_v8
3,HLA-A,ENSG00000206503.12,Artery_Aorta,918.296,gtex_v8
4,HLA-A,ENSG00000206503.12,Artery_Coronary,948.288,gtex_v8
5,HLA-A,ENSG00000206503.12,Artery_Tibial,727.771,gtex_v8
6,HLA-A,ENSG00000206503.12,Bladder,773.672,gtex_v8
7,HLA-A,ENSG00000206503.12,Brain_Amygdala,168.985,gtex_v8
8,HLA-A,ENSG00000206503.12,Brain_Anterior_cingulate_cortex_BA24,131.790,gtex_v8
9,HLA-A,ENSG00000206503.12,Brain_Caudate_basal_ganglia,168.245,gtex_v8


## Part B — GTEx Expression Query

Uses `/expression/clusteredMedianGeneExpression`, which returns median TPM across all
tissues for a gene in one call, avoiding the need to enumerate exact GTEx tissue-site
IDs in advance.

In [8]:
_required = ["gtex_df", "TARGET_COMPARTMENTS"]
_missing = [name for name in _required if name not in dir()]
if _missing:
    raise RuntimeError(f"Missing: {_missing}. Run the previous cell first.")

if gtex_df.empty:
    print("[BLOCKED] No GTEx data retrieved -- check network access and gene resolution above.")
else:
    # Match real GTEx tissue_site_detail_id values to our target compartments,
    # rather than assuming exact naming -- GTEx uses underscored site IDs like
    # "Lung", "Spleen", "Cells_EBV-transformed_lymphocytes", etc.
    TISSUE_SEARCH_TERMS = {
        "lung": ["lung"],
        "lymph_node": ["lymph"],
        "spleen": ["spleen"],
        "upper_airway": ["nasal", "salivary", "trachea", "nose"],
    }

    # GTEx's real ~54-55 tissue sites do NOT include a true solid lymph node
    # organ sample, nor a true upper-airway/nasal-epithelium sample -- confirmed
    # directly against GTEx's own tissue-sampling documentation. What GTEx does
    # have that superficially matches these compartments via substring search:
    #   - "Cells - EBV-transformed lymphocytes" for "lymph" -- a cultured cell
    #     line, not lymph node tissue.
    #   - "Minor_Salivary_Gland" for "salivary" -- an exocrine gland, not
    #     airway epithelium (nasal cavity / nasopharynx / trachea lining).
    # Both would misrepresent the wrong tissue type as the target compartment,
    # so both are explicitly excluded here rather than silently accepted by
    # the substring match.
    EXCLUDE_FALSE_MATCHES = {
        "lymph_node": ["EBV", "transformed", "Cells_"],
        "upper_airway": ["Salivary"],
    }

    available_tissues = gtex_df["tissue_site_detail_id"].unique()
    compartment_matches = {}
    for compartment, terms in TISSUE_SEARCH_TERMS.items():
        matches = [t for t in available_tissues
                  if any(term.lower() in t.lower() for term in terms)]
        exclude_terms = EXCLUDE_FALSE_MATCHES.get(compartment, [])
        matches = [t for t in matches
                  if not any(ex.lower() in t.lower() for ex in exclude_terms)]
        compartment_matches[compartment] = matches
        status = matches if matches else "NO MATCH FOUND"
        print(f"  {compartment}: {status}")

    no_match = [c for c, m in compartment_matches.items() if not m]
    if no_match:
        print(f"\n[WARNING] No GTEx tissue found for: {no_match}.")
        if "lymph_node" in no_match:
            print("  lymph_node: GTEx's real tissue panel does not include a solid "
                  "lymph node organ sample. 'Cells - EBV-transformed lymphocytes' "
                  "superficially matches a 'lymph' text search but is a cultured cell "
                  "line, not lymph node tissue, and has been deliberately excluded "
                  "rather than substituted silently. State this as a genuine GTEx "
                  "data-availability gap in the manuscript -- HPA's lymph node IHC "
                  "data (Part C) may be the only real lymph-node-tissue source "
                  "available for this compartment.")
        if "upper_airway" in no_match:
            print("  upper_airway: GTEx does not sample a true upper-airway/nasal-"
                  "epithelium site directly. 'Minor_Salivary_Gland' superficially "
                  "matches a 'salivary' text search but is exocrine gland tissue, "
                  "not airway epithelium, and has been deliberately excluded rather "
                  "than substituted silently. State this as a genuine GTEx "
                  "data-availability limitation in the manuscript -- HPA (Part C) "
                  "may be the only real upper-airway-tissue source available for "
                  "this compartment.")

    gtex_compartment_rows = []
    for compartment, matched_tissues in compartment_matches.items():
        for tissue in matched_tissues:
            subset = gtex_df[gtex_df["tissue_site_detail_id"] == tissue]
            for _, row in subset.iterrows():
                gtex_compartment_rows.append({**row.to_dict(), "compartment": compartment})

    gtex_compartment_df = pd.DataFrame(gtex_compartment_rows)
    print(f"\n{len(gtex_compartment_df)} GTEx entries matched to target compartments")
    gtex_compartment_df

  lung: ['Lung']
  lymph_node: NO MATCH FOUND
  spleen: ['Spleen']
  upper_airway: NO MATCH FOUND

[WARNING] No GTEx tissue found for: ['lymph_node', 'upper_airway'].
  lymph_node: GTEx's real tissue panel does not include a solid lymph node organ sample. 'Cells - EBV-transformed lymphocytes' superficially matches a 'lymph' text search but is a cultured cell line, not lymph node tissue, and has been deliberately excluded rather than substituted silently. State this as a genuine GTEx data-availability gap in the manuscript -- HPA's lymph node IHC data (Part C) may be the only real lymph-node-tissue source available for this compartment.
  upper_airway: GTEx does not sample a true upper-airway/nasal-epithelium site directly. 'Minor_Salivary_Gland' superficially matches a 'salivary' text search but is exocrine gland tissue, not airway epithelium, and has been deliberately excluded rather than substituted silently. State this as a genuine GTEx data-availability limitation in the manuscript

## Part C — Human Protein Atlas: Real Tissue Vocabulary + HLA Extraction

Downloads the real `normal_tissue.tsv` and inspects its actual tissue names directly,
rather than assuming which names exist.

In [11]:
import zipfile
import io
import requests
import pandas as pd

CANDIDATE_HPA_URLS = [
    "https://www.proteinatlas.org/download/normal_ihc_tissues.tsv.zip",  # per-tissue IHC file (named on the tissue/data page)
    "https://www.proteinatlas.org/download/proteinatlas.tsv.zip",        # master atlas file (confirmed on /about/download)
]

working_url = None
for url in CANDIDATE_HPA_URLS:
    try:
        r = requests.get(url, timeout=120, stream=True)
        print(f"{url}\n  -> status {r.status_code}")
        if r.status_code == 200:
            content = r.content
            try:
                with zipfile.ZipFile(io.BytesIO(content)) as z:
                    inner_name = z.namelist()[0]
                    with z.open(inner_name) as f:
                        df_preview = pd.read_csv(f, sep="\t", nrows=5)
                print(f"  inner file: {inner_name}")
                print(f"  columns ({len(df_preview.columns)}): {list(df_preview.columns)}")
                print(f"  sample row:\n{df_preview.head(1).to_dict('records')}")
                working_url = url
                break
            except zipfile.BadZipFile:
                print("  -> got 200 but response isn't a valid zip (likely an HTML error/redirect page)")
    except requests.RequestException as e:
        print(f"{url}\n  -> request failed: {e}")
    print()

if working_url is None:
    print("[BLOCKED] Neither candidate URL worked. Open "
          "https://www.proteinatlas.org/about/download in a browser, find the "
          "'Protein expression (IHC) | 45 | ...' row, right-click its download "
          "link/icon, copy the exact URL, and paste it back so I can wire it in directly.")
else:
    print(f"\n[OK] Working URL: {working_url}")

https://www.proteinatlas.org/download/normal_ihc_tissues.tsv.zip
  -> status 404

https://www.proteinatlas.org/download/proteinatlas.tsv.zip
  -> status 200
  inner file: proteinatlas.tsv
  columns (119): ['Gene', 'Gene synonym', 'Ensembl', 'Gene description', 'Uniprot', 'Chromosome', 'Position', 'Protein class', 'Biological process', 'Molecular function', 'Disease involvement', 'Evidence', 'HPA evidence', 'UniProt evidence', 'NeXtProt evidence', 'RNA tissue specificity', 'RNA tissue distribution', 'RNA tissue specificity score', 'RNA tissue specific nTPM', 'RNA single cell type specificity', 'RNA single cell type distribution', 'RNA single cell type specificity score', 'RNA single cell type specific nCPM', 'RNA single cell type group specificity', 'RNA single cell type group distribution', 'RNA single cell type group specificity score', 'RNA single cell type group specific nCPM', 'RNA single nuclei brain specificity', 'RNA single nuclei brain distribution', 'RNA single nuclei brain sp

In [12]:
import zipfile
import io
import requests
import pandas as pd

HPA_MASTER_URL = "https://www.proteinatlas.org/download/proteinatlas.tsv.zip"

r = requests.get(HPA_MASTER_URL, timeout=180, stream=True)
r.raise_for_status()
with zipfile.ZipFile(io.BytesIO(r.content)) as z:
    with z.open(z.namelist()[0]) as f:
        hpa_master_df = pd.read_csv(f, sep="\t")

print(f"Master file loaded: {len(hpa_master_df)} rows, {hpa_master_df.shape[1]} columns")

hla_rows = hpa_master_df[hpa_master_df["Gene"].isin(HLA_GENES)]
print(f"\nMatched {len(hla_rows)}/{len(HLA_GENES)} HLA genes by symbol: "
      f"{sorted(hla_rows['Gene'].tolist())}")
missing = set(HLA_GENES) - set(hla_rows["Gene"])
if missing:
    print(f"[WARNING] Not found under these exact symbols: {sorted(missing)} "
          f"-- may be listed under a synonym in 'Gene synonym'.")

cols_of_interest = [
    "Gene",
    "Protein tissue specificity",
    "Protein tissue distribution",
    "Protein tissue specific Intensity",
]
pd.set_option("display.max_colwidth", None)
print("\n" + hla_rows[cols_of_interest].to_string(index=False))

Master file loaded: 20162 rows, 119 columns

Matched 8/8 HLA genes by symbol: ['HLA-A', 'HLA-B', 'HLA-C', 'HLA-DPA1', 'HLA-DPB1', 'HLA-DRA', 'HLA-DRB1', 'HLA-E']

    Gene Protein tissue specificity Protein tissue distribution                                     Protein tissue specific Intensity
HLA-DRB1             Group enriched            Detected in many    intestine: 25876464.9;lung: 36529406.4;lymphoid tissue: 37667592.9
 HLA-DRA             Group enriched             Detected in all intestine: 111164897.2;lung: 164250826.7;lymphoid tissue: 123043087.1
   HLA-C             Group enriched            Detected in many                      intestine: 7422190.3;lymphoid tissue: 17009562.8
   HLA-E            Tissue enhanced            Detected in many                                           lymphoid tissue: 19305177.4
   HLA-A     Low tissue specificity             Detected in all                                                                   NaN
HLA-DPB1             Group enrich

In [13]:
rna_cols_of_interest = [
    "Gene",
    "RNA tissue specificity",
    "RNA tissue distribution",
    "RNA tissue specific nTPM",
]
pd.set_option("display.max_colwidth", None)
print(hla_rows[rna_cols_of_interest].to_string(index=False))

    Gene RNA tissue specificity RNA tissue distribution            RNA tissue specific nTPM
HLA-DRB1        Tissue enhanced        Detected in many  lung: 554.1;lymphoid tissue: 574.2
 HLA-DRA        Tissue enhanced        Detected in many lung: 931.1;lymphoid tissue: 1196.6
   HLA-C        Tissue enhanced        Detected in many             lymphoid tissue: 1377.9
   HLA-E Low tissue specificity        Detected in many                                 NaN
   HLA-A Low tissue specificity        Detected in many                                 NaN
HLA-DPB1        Tissue enhanced        Detected in many  lung: 261.7;lymphoid tissue: 376.0
HLA-DPA1        Tissue enhanced         Detected in all  lung: 516.7;lymphoid tissue: 570.0
   HLA-B        Tissue enhanced        Detected in many             lymphoid tissue: 2350.0


In [ ]:
_required = ["gene_resolution"]
_missing = [name for name in _required if name not in dir()]
if _missing:
    raise RuntimeError(f"Missing: {_missing}. Run Part A first.")

import xml.etree.ElementTree as ET

# PIVOT: HPA's per-gene XML has no tissue-level protein IHC block. What it
# has is ONE rnaExpression element per assayType, each wrapping multiple
# <data> children (one per tissue) -- NOT one rnaExpression per tissue.
# Confirmed structure per <data>:
#   <tissue organ="..." ontologyTerms="...">Adipose tissue</tissue>
#   <level type="normalizedRNAExpression"  unitRNA="nTPM" expRNA="174.9" />
#   <level type="proteinCodingRNAExpression" unitRNA="pTPM" expRNA="243.9" />
#   <level type="RNAExpression"            unitRNA="TPM"  expRNA="193.9" />
#   <RNASample .../> * N   -- individual donor samples, not used here
# assayType="tissue" = HPA's own independent RNA-seq (used below).
# assayType="consensusTissue" = max(HPA, GTEx) -- NOT used, would be
# circular against Part B.

ensembl_ids = {gene: gid.split(".")[0] for gene, gid in gene_resolution.items() if gid}

test_gene = "HLA-A"
test_id = ensembl_ids.get(test_gene)
url = f"https://www.proteinatlas.org/{test_id}.xml"
r = requests.get(url, timeout=60)
print(f"Status code: {r.status_code}, size: {len(r.content)} bytes\n")

if r.status_code != 200:
    print(f"[BLOCKED] Could not fetch XML (status {r.status_code}).")
else:
    root = ET.fromstring(r.content)
    rna_tissue_elem = next((e for e in root.iter("rnaExpression") if e.get("assayType") == "tissue"), None)

    if rna_tissue_elem is None:
        print("[BLOCKED] No assayType='tissue' rnaExpression element found.")
    else:
        data_children = rna_tissue_elem.findall("data")
        print(f"{len(data_children)} <data> (tissue) entries found for {test_gene}\n")
        print("--- First 3 tissue entries, parsed ---")
        for d in data_children[:3]:
            tissue_elem = d.find("tissue")
            tissue_name = (tissue_elem.text or "").strip() if tissue_elem is not None else None
            levels = {lvl.get("unitRNA"): lvl.get("expRNA") for lvl in d.findall("level")}
            print(f"  tissue={tissue_name!r}  nTPM={levels.get('nTPM')}  "
                  f"pTPM={levels.get('pTPM')}  TPM={levels.get('TPM')}")


In [ ]:
_required = ["ensembl_ids", "HLA_GENES", "TARGET_COMPARTMENTS"]
_missing = [name for name in _required if name not in dir()]
if _missing:
    raise RuntimeError(f"Missing: {_missing}. Run the previous cell first.")

import xml.etree.ElementTree as ET

def parse_data_entry(data_elem) -> dict:
    tissue_elem = data_elem.find("tissue")
    tissue_name = (tissue_elem.text or "").strip() if tissue_elem is not None else None
    organ = tissue_elem.get("organ") if tissue_elem is not None else None
    levels = {lvl.get("unitRNA"): lvl.get("expRNA") for lvl in data_elem.findall("level")}
    return {
        "tissue": tissue_name,
        "organ": organ,
        "nTPM": levels.get("nTPM"),
        "pTPM": levels.get("pTPM"),
        "TPM": levels.get("TPM"),
    }

def fetch_hpa_rna_tissue(gene_symbol: str, ensembl_id: str) -> list:
    url = f"https://www.proteinatlas.org/{ensembl_id}.xml"
    try:
        r = requests.get(url, timeout=60)
        r.raise_for_status()
    except requests.RequestException as e:
        print(f"  [WARN] {gene_symbol} ({ensembl_id}): request failed: {e}")
        return []

    root = ET.fromstring(r.content)
    rna_tissue_elem = next((e for e in root.iter("rnaExpression") if e.get("assayType") == "tissue"), None)
    if rna_tissue_elem is None:
        print(f"  [WARN] {gene_symbol}: no assayType='tissue' element found")
        return []

    rows = []
    for data_elem in rna_tissue_elem.findall("data"):
        parsed = parse_data_entry(data_elem)
        parsed["gene"] = gene_symbol
        parsed["ensembl_id"] = ensembl_id
        rows.append(parsed)
    return rows

hpa_rna_rows = []
for gene, ensembl_id in ensembl_ids.items():
    entries = fetch_hpa_rna_tissue(gene, ensembl_id)
    print(f"  {gene}: {len(entries)} tissue entries returned")
    hpa_rna_rows.extend(entries)

hpa_rna_df = pd.DataFrame(hpa_rna_rows)
print(f"\nTotal HPA RNA-tissue entries: {len(hpa_rna_df)}")

if hpa_rna_df.empty:
    print("[BLOCKED] No HPA RNA-tissue data retrieved.")
    hpa_hla_df = pd.DataFrame(columns=["gene", "tissue", "compartment", "nTPM", "pTPM", "TPM", "assay_type"])
else:
    real_hpa_tissues = sorted(hpa_rna_df["tissue"].dropna().unique())
    print(f"\n{len(real_hpa_tissues)} real HPA tissue names found: {real_hpa_tissues}")

    HPA_SEARCH_TERMS = {
        "lung": ["lung"],
        "lymph_node": ["lymph node"],
        "spleen": ["spleen"],
        "upper_airway": ["nasopharynx", "bronchus", "trachea", "nasal"],
    }

    hpa_compartment_matches = {}
    for compartment, terms in HPA_SEARCH_TERMS.items():
        matches = [t for t in real_hpa_tissues
                  if any(term.lower() in t.lower() for term in terms)]
        hpa_compartment_matches[compartment] = matches
        status = matches if matches else "NO MATCH FOUND in real HPA vocabulary"
        print(f"  {compartment}: {status}")

    no_match_hpa = [c for c, m in hpa_compartment_matches.items() if not m]
    if no_match_hpa:
        print(f"\n[NOTE] No real HPA tissue matched for: {no_match_hpa}. State this "
              f"as a real data-availability limitation, not a substituted proxy.")

    hpa_hla_rows = []
    for compartment, matched_tissues in hpa_compartment_matches.items():
        for tissue in matched_tissues:
            subset = hpa_rna_df[(hpa_rna_df["gene"].isin(HLA_GENES)) &
                                 (hpa_rna_df["tissue"] == tissue)]
            for _, row in subset.iterrows():
                hpa_hla_rows.append({
                    "gene": row["gene"], "tissue": tissue, "compartment": compartment,
                    "nTPM": row["nTPM"], "pTPM": row["pTPM"], "TPM": row["TPM"],
                    "assay_type": "tissue",
                })

    hpa_hla_df = pd.DataFrame(hpa_hla_rows)
    print(f"\n{len(hpa_hla_df)} HPA HLA RNA-tissue entries extracted for target compartments")

hpa_hla_df.head(15)


## Part D — Merge, Visualize, Export

In [ ]:
_required = ["gtex_compartment_df", "hpa_hla_df", "FRAMING_NOTE"]
_missing = [name for name in _required if name not in dir()]
if _missing:
    raise RuntimeError(f"Missing: {_missing}. Run Parts B and C first.")

# PIVOT NOTE (see Part C): hpa_hla_df now holds HPA's independent per-tissue
# RNA-seq (assayType="tissue"), not protein IHC. Cast expRNA strings to
# numeric before plotting -- HPA's XML returns nTPM/pTPM/TPM as text.
for col in ["nTPM", "pTPM", "TPM"]:
    if col in hpa_hla_df.columns:
        hpa_hla_df[col] = pd.to_numeric(hpa_hla_df[col], errors="coerce")

PIVOT_NOTE = (
    "Part C pivoted from HPA protein IHC (unavailable in HPA's current export "
    "surface -- confirmed absent from both bulk TSV and per-gene XML) to HPA's "
    "independent tissue RNA-seq (assayType='tissue'). Part C is therefore a "
    "same-modality, cross-pipeline replication of Part B (GTEx RNA-seq vs. "
    "HPA RNA-seq), not the originally planned protein-vs-RNA cross-modality "
    "check. The 'consensusTissue' assayType (max(HPA,GTEx)) was deliberately "
    "excluded to avoid circularity against Part B."
)

# --- GTEx heatmap: gene x compartment, mean median TPM ---
if not gtex_compartment_df.empty:
    gtex_pivot = gtex_compartment_df.pivot_table(
        index="gene", columns="compartment", values="median_tpm", aggfunc="mean")

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(gtex_pivot, annot=True, fmt=".1f", cmap="YlOrRd", ax=ax,
                linewidths=0.5, cbar_kws={"label": "Median TPM (GTEx)"})
    ax.set_title("HLA Gene Tissue-Compartment Expression\n(GTEx, current release, median TPM)")
    ax.set_xlabel("Tissue compartment")
    ax.set_ylabel("HLA gene")
    plt.tight_layout()

    gtex_fig_path = FIGURES_DIR / "Figure_gtex_hla_tissue_expression.png"
    plt.savefig(gtex_fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {gtex_fig_path}")
else:
    gtex_fig_path = None
    print("[INFO] No GTEx compartment data to plot.")

# --- HPA RNA heatmap: gene x compartment, mean nTPM (independent HPA RNA-seq) ---
if not hpa_hla_df.empty:
    hpa_pivot = hpa_hla_df.pivot_table(
        index="gene", columns="compartment", values="nTPM", aggfunc="mean")

    fig, ax = plt.subplots(figsize=(8, 5))
    sns.heatmap(hpa_pivot, annot=True, fmt=".1f", cmap="YlGnBu", ax=ax,
                linewidths=0.5, cbar_kws={"label": "nTPM (HPA, assayType='tissue')"})
    ax.set_title("HLA Gene Tissue-Compartment Expression\n(HPA independent RNA-seq, nTPM)")
    ax.set_xlabel("Tissue compartment")
    ax.set_ylabel("HLA gene")
    plt.tight_layout()

    hpa_fig_path = FIGURES_DIR / "Figure_hpa_hla_tissue_expression.png"
    plt.savefig(hpa_fig_path, dpi=200, bbox_inches="tight")
    plt.show()
    print(f"Saved: {hpa_fig_path}")

    if "upper_airway" not in hpa_pivot.columns:
        print("[NOTE] 'upper_airway' column absent from HPA pivot -- no matching "
              "tissue in HPA's real vocabulary either (consistent with GTEx).")
else:
    hpa_fig_path = None
    print("[INFO] No HPA RNA-tissue data to plot.")

# --- Export tables ---
gtex_out_path = TABLES_DIR / "Table_S6_gtex_hla_expression.csv"
hpa_out_path = TABLES_DIR / "Table_S6_hpa_hla_expression.csv"
gtex_compartment_df.to_csv(gtex_out_path, index=False)
hpa_hla_df.to_csv(hpa_out_path, index=False)

print(f"\nSaved: {gtex_out_path}")
print(f"Saved: {hpa_out_path}")

try:
    from google.colab import files
    files.download(str(gtex_out_path))
    files.download(str(hpa_out_path))
    if gtex_fig_path is not None:
        files.download(str(gtex_fig_path))
    if hpa_fig_path is not None:
        files.download(str(hpa_fig_path))
except ImportError:
    print("Files saved locally.")

print(f"\n{FRAMING_NOTE}")
print(f"\n{PIVOT_NOTE}")
print("\nStep 9 complete. Feed both tables into Step 10 (final integration).")
print("\nIMPORTANT: any 'upper_airway' compartment gaps flagged above (GTEx and/or HPA) "
      "must be stated explicitly as a data-availability limitation in the manuscript -- "
      "neither database samples a canonical 'upper airway' site directly, and any "
      "proxy tissue used should be named and justified explicitly, not silently assumed.")
